# Практика №6.

## Задание

1. Напишите вспомогательную программу построения таблицы простых чисел меньших 256 с помощью решета Эратосфена.
2. Выясните с помощью метода Ферма, являются ли n произвольных чисел простыми; в случае составного числа разложите его на множители.
3. Для произвольного большого простого числа p выясните вопрос о его простоте:
   - с помощью теста Соловея – Штрассена;
   - с помощью теста Лемана;
   - с помощью теста Рабина – Миллера;
   - с помощью непосредственной проверки.

In [73]:
import math
import random
import numpy as np

### 1. Напишите вспомогательную программу построения таблицы простых чисел меньших 256 с помощью решета Эратосфена.

In [74]:
def sieve_of_eratosthenes(limit):
    # Создаем булев массив
    is_prime = np.ones(limit + 1, dtype=bool)
    is_prime[0:2] = False

    for num in range(2, int(limit**0.5) + 1):
        if is_prime[num]:
            # Обнуляем все кратные num, начиная с его квадрата
            is_prime[num*num:limit+1:num] = False

    # Собираем все простые числа
    primes = np.nonzero(is_prime)[0]
    return primes

In [75]:
limit = 255 # Предел для поиска простых чисел

# Построение таблицы простых чисел
primes = sieve_of_eratosthenes(limit)
print(primes)

[  2   3   5   7  11  13  17  19  23  29  31  37  41  43  47  53  59  61
  67  71  73  79  83  89  97 101 103 107 109 113 127 131 137 139 149 151
 157 163 167 173 179 181 191 193 197 199 211 223 227 229 233 239 241 251]


### 2. Выясните с помощью метода Ферма, являются ли n произвольных чисел простыми; в случае составного числа разложите его на множители.

In [76]:
def fermat_factorization(n):
    if n <= 1:
        return (1, n)
    if n % 2 == 0:
        return (n // 2, 2)

    x = math.isqrt(n) + 1 
    y = 0
    while True:
        k = x * x - n
        y = math.isqrt(k)
        
        if y * y == k:
            p = x - y
            q = x + y
            return (p, q)

        x += 1

In [77]:
for num in np.arange(0, 20):
    fermat_test_out = fermat_factorization(num)
    if fermat_test_out[0] == 1:
        print(num)
        continue
    print(f"{num} = {fermat_test_out[0]} * {fermat_test_out[1]}")

0
1
2
3
4 = 2 * 2
5
6 = 3 * 2
7
8 = 4 * 2
9
10 = 5 * 2
11
12 = 6 * 2
13
14 = 7 * 2
15 = 3 * 5
16 = 8 * 2
17
18 = 9 * 2
19


### 3. Для произвольного большого простого числа p выясните вопрос о его простоте

- с помощью теста Соловея – Штрассена;
- с помощью теста Лемана;
- с помощью теста Рабина – Миллера;
- с помощью непосредственной проверки.

Тест Соловея – Штрассена

In [78]:
def jacobi_symbol(a, p):
    if a % p == 0:
        return 0
    result = 1
    if a < 0:
        a = -a
        if p % 4 == 3:
            result = -result

    while a != 0:
        while a % 2 == 0:
            a //= 2
            if p % 8 in [3, 5]:
                result = -result
        a, p = p, a
        if a % 4 == 3 and p % 4 == 3:
            result = -result
        a %= p

    if p == 1:
        return result
    return 0

In [79]:
def solovay_strassen(test_number, iterations=5):
    if test_number < 2:
        return False
    if test_number in (2, 3):
        return True
    if test_number % 2 == 0:
        return False

    for _ in range(iterations):
        random_base = random.randint(1, test_number - 1)
        if math.gcd(random_base, test_number) != 1:
            return False

        quadratic_residue = pow(random_base, (test_number - 1) // 2, test_number)
        jacobi_value = jacobi_symbol(random_base, test_number)

        if quadratic_residue != jacobi_value % test_number:
            return False

    return True

Тест Лемана

In [80]:
def lehmann(test_number, iterations=5):
    if test_number < 2:
        return False
    if test_number in (2, 3):
        return True
    if test_number % 2 == 0:
        return False

    for _ in range(iterations):
        random_base = random.randint(1, test_number - 1)
        result = pow(random_base, (test_number - 1) // 2, test_number)

        if result not in [1, test_number - 1]:
            return False

    return True

Тест Рабина – Миллера

In [81]:
def rabin_miller(test_number, iterations=5):
    if test_number < 2:
        return False
    if test_number in (2, 3):
        return True
    if test_number % 2 == 0:
        return False

    m = test_number - 1
    power_of_two = 0
    while m % 2 == 0:
        m //= 2
        power_of_two += 1

    for _ in range(iterations):
        random_base = random.randint(1, test_number - 1)
        residue = pow(random_base, m, test_number)

        if residue == 1 or residue == test_number - 1:
            continue

        for _ in range(power_of_two - 1):
            residue = (residue * residue) % test_number
            if residue == test_number - 1:
                break
        else:
            return False

    return True

Непосредственная проверка

In [82]:
def is_prime_check(n):
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False

    for i in range(5, int(math.sqrt(n)) + 1, 6):
        if n % i == 0 or n % (i + 2) == 0:
            return False

    return True

In [83]:
number = 104729

is_prime_solovay = solovay_strassen(number)
is_prime_lehmann = lehmann(number)
is_prime_rabin = rabin_miller(number)
is_prime = is_prime_check(number)

print(f"Число {number} (Соловей-Штрассен): {'простое' if is_prime_solovay else 'не простое'}.")
print(f"Число {number} (Леманн): {'простое' if is_prime_lehmann else 'не простое'}.")
print(f"Число {number} (Рабин-Миллер): {'простое' if is_prime_rabin else 'не простое'}.")
print(f"Число {number}: {'простое' if is_prime else 'не простое'}.")

Число 104729 (Соловей-Штрассен): простое.
Число 104729 (Леманн): простое.
Число 104729 (Рабин-Миллер): простое.
Число 104729: простое.


In [84]:
n = 2048

for num in range(2 ** (n-1), 2 ** n):
    is_prime = solovay_strassen(num)
    if is_prime:
        print(num)
        break


16158503035655503650357438344334975980222051334857742016065172713762327569433945446598600705761456731844358980460949009747059779575245460547544076193224141560315438683650498045875098875194826053398028819192033784138396109321309878080919047169238085235290822926018152521443787945770532904303776199561965192760957166694834171210342487393282284747428088017663161029038902829665513096354230157075129296432088558362971801859230928678799175576150822952201848806616643615613562842355410104862578550863465661734839271290328348967522998634176499319107762583194718667771801067716614802322659239302476074096777926805529798117247


In [85]:
for num in range(2 ** (n-1), 2 ** n):
    is_prime = lehmann(num)
    if is_prime:
        print(num)
        break

16158503035655503650357438344334975980222051334857742016065172713762327569433945446598600705761456731844358980460949009747059779575245460547544076193224141560315438683650498045875098875194826053398028819192033784138396109321309878080919047169238085235290822926018152521443787945770532904303776199561965192760957166694834171210342487393282284747428088017663161029038902829665513096354230157075129296432088558362971801859230928678799175576150822952201848806616643615613562842355410104862578550863465661734839271290328348967522998634176499319107762583194718667771801067716614802322659239302476074096777926805529798117247


In [86]:
for num in range(2 ** (n-1), 2 ** n):
    is_prime = rabin_miller(num)
    if is_prime:
        print(num)
        break

16158503035655503650357438344334975980222051334857742016065172713762327569433945446598600705761456731844358980460949009747059779575245460547544076193224141560315438683650498045875098875194826053398028819192033784138396109321309878080919047169238085235290822926018152521443787945770532904303776199561965192760957166694834171210342487393282284747428088017663161029038902829665513096354230157075129296432088558362971801859230928678799175576150822952201848806616643615613562842355410104862578550863465661734839271290328348967522998634176499319107762583194718667771801067716614802322659239302476074096777926805529798117247


In [88]:
for num in range(2 ** (n-1), 2 ** n):
    is_prime = is_prime_check(num)
    if is_prime:
        print(num)
        break

OverflowError: int too large to convert to float